# Coleta de produtos por nome — versão 43

Notebook histórico da coleta concluída. A lógica original foi preservada integralmente; não é necessário executá-lo novamente.

O resultado definitivo está em `dados/produtos_por_nome_anvisa.xlsx`. Os nomes de caminhos usados durante a execução original permanecem no código para preservar sua procedência.

Para analisar a base entregue, use `analisar_produtos_por_nome.R`. As saídas de execução incorporadas ao notebook foram limpas apenas para facilitar a leitura.

# ANVISA Scraper v43

**v43** — transient server errors (e.g. HTTP 500) no longer get silently recorded as "0 records". Each category is now retried with a **fresh session** and escalating backoff (`CATEGORY_MAX_ATTEMPTS`), individual pages are retried within a category (`PAGE_RETRY_ROUNDS`), and a term is **only written to the checkpoint if every category succeeded**. Terms that still fail are collected and retried in extra passes at the end of the run (`RETRY_PASSES`); anything still failing is reported and left out of the checkpoint so a simple re-run picks it up. See the new retry knobs in cell 2.

**v42** — reads search terms from `PlantasBuscar.csv`. This file is **semicolon-delimited** (`sep=';'`) and uses different column names than before: `Nome_popular_buscado`, `Nome_cientifico_buscado`, `Variacoes_nome_popular_encontradas`, `Variacoes_nome_cientifico_encontradas`. Term extraction is otherwise unchanged: each source column is split on `;` and `/`, `NA` tokens are dropped (case-insensitive), and terms are deduplicated globally before scraping.

**v41** — read search terms from `plantasfito.csv` (comma-delimited; columns `Nome popular`, `Nome científico`, `Variacoes_nome_popular_encontradas`, `Variacoes_nome_cientifico_encontradas`). Splits on `;` and `/`, drops `NA` tokens (case-insensitive), deduplicates globally before scraping.

**v40** — stricter term cleaning: every character that isn't a letter or whitespace is stripped. Removes botanical author abbreviations (`acinax raf.` → `acinax raf`), hybrid markers (`rosmarinus ×lavandulaceus` → `rosmarinus lavandulaceus`), apostrophes, digits, and punctuation from the search terms.

**v38** — search terms now come from columns I–N of `plantas_fitoterapicas_enriquecido.csv` (split on `|`, plus `,` inside a piece; trailing `[lang]` tags stripped from col M). Configurable via `SEARCH_COLUMNS`, `LATIN_ONLY`, `MAX_TERMS` in cell 2.

**v36** adds post-scraping filters as a dedicated cell (cell 5), applied to `df_all` before export:
- **`SKIP_TERMS`**: drop all rows for specific search terms (e.g. `penicilina`).
- **Word-boundary filter**: drop rows where the search term is embedded inside another word (e.g. `alho` inside `trabalho`).

**Inherited from v33:** deep flatten, drop empty columns per sheet.

## 1 · Imports & config

In [ ]:
import urllib.request, urllib.parse, http.cookiejar
import json, time, threading, re
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from IPython.display import display

BASE = 'https://consultas.anvisa.gov.br'

# ── Search terms from PlantasBuscar.csv ───────────────────────────────────
# NOTE: PlantasBuscar.csv is SEMICOLON-delimited -> pd.read_csv(..., sep=';').
# Columns used as search-term sources:
#   Nome_popular_buscado                  — canonical popular name (may contain "/" separating variants)
#   Nome_cientifico_buscado               — canonical scientific name (may contain "/" separating variants)
#   Variacoes_nome_popular_encontradas    — popular name variants, ";"-separated (may contain "NA")
#   Variacoes_nome_cientifico_encontradas — scientific name variants, ";"-separated (may contain "NA")
#
# Each column is split on ";" and "/" into individual terms.
# "NA" tokens (case-insensitive, after stripping) are dropped.
# Produces df_terms with columns: scientific, search_term.
# SEARCH_TERMS is the globally dedup'd list the scraper iterates.

CSV_PATH = 'PlantasBuscar.csv'

# Set True to skip scientific-name columns (search popular names only)
POPULAR_ONLY = False

MIN_TERM_LENGTH = 3        # drop very short tokens that would be noisy
LATIN_ONLY      = True     # skip non-Latin scripts
MAX_TERMS       = None     # cap for dry-runs; set e.g. 50 to smoke-test

_LATIN_OK = re.compile(r'^[\sa-zA-ZÀ-ɏ]+$')

def _clean(name: str) -> str:
    """Keep only letters and whitespace; collapse runs of spaces."""
    s = ''.join(c if (c.isalpha() or c.isspace()) else ' ' for c in str(name))
    s = re.sub(r'\s+', ' ', s).strip()
    return s.lower()

def _split_cell(value: str):
    """Split a cell on ';' and '/', strip each piece, drop 'NA' tokens."""
    pieces = []
    for part in re.split(r'[;/]', str(value)):
        part = part.strip()
        if part.upper() == 'NA' or part == '':
            continue
        pieces.append(part)
    return pieces

def _build_term_df(csv_path=CSV_PATH, popular_only=POPULAR_ONLY,
                   min_len=MIN_TERM_LENGTH, latin_only=LATIN_ONLY):
    df = pd.read_csv(csv_path, sep=';')

    # Columns that supply the canonical "row label" (scientific name used as key)
    LABEL_COL = 'Nome_cientifico_buscado'
    # Columns whose values become search terms
    SEARCH_COLUMNS = [
        'Nome_popular_buscado',
        'Variacoes_nome_popular_encontradas',
    ]
    if not popular_only:
        SEARCH_COLUMNS += [
            'Nome_cientifico_buscado',
            'Variacoes_nome_cientifico_encontradas',
        ]

    missing = [c for c in [LABEL_COL] + SEARCH_COLUMNS if c not in df.columns]
    if missing:
        raise KeyError(f'Missing columns in {csv_path}: {missing}')

    rows = []
    for _, r in df.iterrows():
        sci = str(r[LABEL_COL]).strip()
        # Per-row dedup: the same cleaned term won't repeat within one species
        seen_in_row = set()
        for col in SEARCH_COLUMNS:
            val = r[col]
            if pd.isna(val):
                continue
            for piece in _split_cell(str(val)):
                t = _clean(piece)
                if len(t) < min_len:
                    continue
                if latin_only and not _LATIN_OK.match(t):
                    continue
                if t in seen_in_row:
                    continue
                seen_in_row.add(t)
                rows.append((sci, t))

    return pd.DataFrame(rows, columns=['scientific', 'search_term'])

df_terms = _build_term_df()

# Globally dedup'd list (same term from multiple species is scraped once)
SEARCH_TERMS = df_terms['search_term'].drop_duplicates().tolist()
if MAX_TERMS is not None:
    SEARCH_TERMS = SEARCH_TERMS[:MAX_TERMS]

print(f'Loaded {len(df_terms)} (scientific, search_term) pairs from {CSV_PATH}')
print(f'  unique scientifics : {df_terms["scientific"].nunique()}')
print(f'  unique search terms: {len(SEARCH_TERMS)}')
print('\nFirst 10 pairs:')
display(df_terms.head(10))

# Page size sent as `count=`. Try 500 first; lower to 200/100 if rejected.
PAGE_SIZE = 500

# Per-request timeout. cosmeticos_regularizados can be very slow.
REQUEST_TIMEOUT = 180
MAX_RETRIES = 3

# Parallelism inside each category (after warmup is done)
PAGES_CONCURRENCY = 6

# Parallelism across categories (each gets its own cookie jar)
CATEGORY_CONCURRENCY = 6

# ── Retry behaviour on transient server errors (e.g. HTTP 500) ────────────
# hit() already retries each individual request MAX_RETRIES times with a
# short backoff. The knobs below add coarser, more patient retries on top:
#   * a failed page 1 -> re-attempt the whole category with a FRESH session
#   * failed pages 2..N -> retried in rounds before the category is re-attempted
#   * a term is checkpointed ONLY if every category succeeded; still-failing
#     terms are retried in extra passes at the end of the run.
CATEGORY_MAX_ATTEMPTS  = 4    # per-category attempts (fresh session each) before giving up
CATEGORY_RETRY_BACKOFF = 8    # base seconds between category attempts (x attempt number)
PAGE_RETRY_ROUNDS      = 3    # retry rounds for individual pages 2..N within a category
PAGE_RETRY_BACKOFF     = 4    # base seconds between page-retry rounds (x round number)
RETRY_PASSES           = 3    # extra full passes over still-failing terms at the run's end
RETRY_PASS_PAUSE       = 30   # seconds to pause before each extra pass

ENDPOINTS = {
    'alimentos':                'alimento/produtos/',
    'cosmeticos_registrados':   'cosmeticos/registrados',
    'cosmeticos_regularizados': 'cosmeticos/regularizados',
    'medicamentos':             'medicamento/produtos/',
    'saneantes':                'saneantes/produtos',
    'produtos_saude':           'saude',
}

WARMUPS = {
    'alimentos': [
        '/api/pais/paises',
        '/api/uf',
        '/api/grupoPopulacional/',
        '/api/viaAdministracao/6',
        '/api/composicaoAlergenico/',
    ],
    'cosmeticos_registrados': [],
    'cosmeticos_regularizados': [
        '/scripts/components/modal-form/modal-form.tpl.html',
        '/scripts/app/main/options.cosmetico.html',
        '/scripts/app/cosmeticos/regularizado/cosmeticos.filtro.html',
        '/scripts/app/components/input-empresa/input-empresa.directive.html',
        '/scripts/components/date/datepicker.anvisa.directive.html',
        '/scripts/app/cosmeticos/regularizado/cosmeticos.result.html',
        '/scripts/app/components/monitoramento/monitoramento.directive.html',
    ],
    'medicamentos':             [],
    'saneantes':                [],
    'produtos_saude':           [],
}

print('Config ready.')


## 2 · Headers

In [ ]:
API_HEADERS = {
    'Accept':              'application/json, text/plain, */*',
    'Accept-Language':     'en-US,en;q=0.9,pt-BR;q=0.8,pt;q=0.7',
    'Authorization':       'Guest',
    'Cache-Control':       'no-cache',
    'DNT':                 '1',
    'If-Modified-Since':   'Mon, 26 Jul 1997 05:00:00 GMT',
    'Pragma':              'no-cache',
    'Referer':             'https://consultas.anvisa.gov.br/',
    'Sec-Fetch-Dest':      'empty',
    'Sec-Fetch-Mode':      'cors',
    'Sec-Fetch-Site':      'same-origin',
    'User-Agent':          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36',
    'sec-ch-ua':           '"Google Chrome";v="147", "Not.A/Brand";v="8", "Chromium";v="147"',
    'sec-ch-ua-mobile':    '?0',
    'sec-ch-ua-platform':  '"Windows"',
    'x-dtreferer':         'https://consultas.anvisa.gov.br/',
}

HTML_HEADERS = dict(API_HEADERS)
HTML_HEADERS['Accept'] = 'text/html'

HOME_HEADERS = {
    'Accept':                    'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language':           API_HEADERS['Accept-Language'],
    'User-Agent':                API_HEADERS['User-Agent'],
    'Sec-Fetch-Dest':            'document',
    'Sec-Fetch-Mode':            'navigate',
    'Sec-Fetch-Site':            'none',
    'Upgrade-Insecure-Requests': '1',
    'sec-ch-ua':                 API_HEADERS['sec-ch-ua'],
    'sec-ch-ua-mobile':          API_HEADERS['sec-ch-ua-mobile'],
    'sec-ch-ua-platform':        API_HEADERS['sec-ch-ua-platform'],
}

print('Headers ready.')

## 3 · Helpers

In [ ]:
# Lock printing so concurrent threads don't garble output
_print_lock = threading.Lock()
def log(msg):
    with _print_lock:
        print(msg)


def hit(opener, url, headers, timeout=None, max_retries=MAX_RETRIES):
    req = urllib.request.Request(url, headers=headers)
    last_err = None
    for attempt in range(max_retries):
        try:
            with opener.open(req, timeout=timeout or REQUEST_TIMEOUT) as r:
                return r.status, r.read()
        except urllib.error.HTTPError as e:
            body = e.read()[:120]
            if e.code in (429,) or e.code >= 500:
                last_err = f'HTTP {e.code}'
                time.sleep(2 ** attempt)
                continue
            return e.code, body
        except Exception as e:
            last_err = str(e)
            time.sleep(2 ** attempt)
    return -1, str(last_err).encode()


def make_opener():
    """Each category gets its own cookie jar / opener for clean parallelism."""
    jar = http.cookiejar.CookieJar()
    return urllib.request.build_opener(urllib.request.HTTPCookieProcessor(jar)), jar


def warmup_homepage(opener):
    status, _ = hit(opener, BASE + '/', HOME_HEADERS)
    return status == 200


def warmup_category(opener, category):
    paths = WARMUPS.get(category, [])
    for path in paths:
        headers = HTML_HEADERS if path.endswith('.html') else API_HEADERS
        hit(opener, BASE + path, headers)


def fetch_search_page(opener, category, page_num, search_term):
    qs = urllib.parse.urlencode({
        'column': '',
        'count': PAGE_SIZE,
        'filter[nomeProduto]': search_term,
        'order': 'asc',
        'page': page_num,
    })
    url = f'{BASE}/api/consulta/{ENDPOINTS[category]}?{qs}'
    status, body = hit(opener, url, API_HEADERS)
    if status != 200:
        body_preview = body[:120] if isinstance(body, bytes) else str(body)[:120]
        log(f'  [{category}|{search_term}] page {page_num} -> {status}  body[:120]={body_preview!r}')
        return None
    try:
        return json.loads(body)
    except json.JSONDecodeError as e:
        log(f'  [{category}|{search_term}] page {page_num} -> 200 but JSON parse failed: {e}')
        return None


def scrape_category(category, search_term):
    """Scrape one category end-to-end for a given search_term.

    Resilient to transient server errors (e.g. HTTP 500):
      * page 1 is retried up to CATEGORY_MAX_ATTEMPTS times, each with a
        FRESH cookie jar + warmup and an escalating backoff;
      * pages 2..N are retried per-page for PAGE_RETRY_ROUNDS rounds, and if
        any page still won't load the whole category is re-attempted.

    Returns (records, ok). ok is False only if every attempt failed or some
    page could never be retrieved. The caller must NOT treat ok=False as a
    genuine 0-result -- it means "unknown, retry later".
    """
    last_reason = 'unknown'
    for attempt in range(1, CATEGORY_MAX_ATTEMPTS + 1):
        t0 = time.time()
        opener, _ = make_opener()

        if not warmup_homepage(opener):
            last_reason = 'homepage warmup failed'
            log(f'[{category}|{search_term}] {last_reason} '
                f'(attempt {attempt}/{CATEGORY_MAX_ATTEMPTS})')
            time.sleep(CATEGORY_RETRY_BACKOFF * attempt)
            continue
        warmup_category(opener, category)

        suffix = '' if attempt == 1 else f' (attempt {attempt}/{CATEGORY_MAX_ATTEMPTS})'
        log(f'[{category}|{search_term}] fetching page 1...{suffix}')
        first = fetch_search_page(opener, category, 1, search_term)
        if first is None:
            last_reason = 'page 1 failed'
            log(f'[{category}|{search_term}] page 1 failed -- will retry '
                f'(attempt {attempt}/{CATEGORY_MAX_ATTEMPTS})')
            time.sleep(CATEGORY_RETRY_BACKOFF * attempt)
            continue

        content = first.get('content') or first.get('data') or first.get('resultado') or []
        total_pages = (first.get('totalPages') or first.get('totalPaginas')
                       or first.get('last_page') or 1)
        total = (first.get('totalElements') or first.get('totalItens')
                 or first.get('total') or len(content))
        log(f'[{category}] {total} records across {total_pages} pages (page_size={PAGE_SIZE})')

        records = [dict(it, _category=category, _search_term=search_term) for it in content]

        # Fetch pages 2..N, retrying any that fail for a few rounds ──────────
        pending = list(range(2, total_pages + 1))
        for round_i in range(1, PAGE_RETRY_ROUNDS + 1):
            if not pending:
                break
            failed_pages = []
            with ThreadPoolExecutor(max_workers=PAGES_CONCURRENCY) as ex:
                futures = {ex.submit(fetch_search_page, opener, category, p, search_term): p
                           for p in pending}
                done_count = 0
                for fut in as_completed(futures):
                    p = futures[fut]
                    page_data = fut.result()
                    done_count += 1
                    if page_data is None:
                        failed_pages.append(p)
                        continue
                    page_content = (page_data.get('content') or page_data.get('data')
                                    or page_data.get('resultado') or [])
                    records.extend(dict(it, _category=category, _search_term=search_term)
                                   for it in page_content)
                    if done_count % 5 == 0 or done_count == len(pending):
                        log(f'  [{category}] {done_count}/{len(pending)} pages done '
                            f'(round {round_i}), {len(records)} records')
            pending = failed_pages
            if pending:
                log(f'  [{category}|{search_term}] {len(pending)} page(s) failed, retrying '
                    f'(round {round_i}/{PAGE_RETRY_ROUNDS}): {pending}')
                time.sleep(PAGE_RETRY_BACKOFF * round_i)

        if pending:
            # Pages still missing with this session -> re-attempt the whole
            # category with a fresh one (outer loop).
            last_reason = f'{len(pending)} page(s) unretrievable: {pending}'
            log(f'[{category}|{search_term}] {last_reason} -- re-attempting category '
                f'(attempt {attempt}/{CATEGORY_MAX_ATTEMPTS})')
            time.sleep(CATEGORY_RETRY_BACKOFF * attempt)
            continue

        elapsed = time.time() - t0
        log(f'[{category}|{search_term}] done -- {len(records)} records in {elapsed:.1f}s')
        return records, True

    log(f'[{category}|{search_term}] GAVE UP after {CATEGORY_MAX_ATTEMPTS} attempts '
        f'(last: {last_reason})')
    return [], False


def scrape_term(search_term):
    """Scrape every category for one term (categories run in parallel).

    Returns (term_flat_records, term_ok, term_record_count, failed_categories).
    term_ok is True only when *all* categories succeeded.
    """
    term_results = {}
    with ThreadPoolExecutor(max_workers=CATEGORY_CONCURRENCY) as ex:
        futures = {ex.submit(scrape_category, cat, search_term): cat
                   for cat in ENDPOINTS}
        for fut in as_completed(futures):
            cat = futures[fut]
            try:
                term_results[cat] = fut.result()
            except Exception as e:
                log(f'!! {cat}|{search_term} raised: {e!r}')
                term_results[cat] = ([], False)

    failed_cats = [c for c, (_recs, ok) in term_results.items() if not ok]
    term_ok = not failed_cats
    raw = [r for (recs, _ok) in term_results.values() for r in recs]
    term_total = sum(len(recs) for (recs, _ok) in term_results.values())
    return flatten_records(raw), term_ok, term_total, failed_cats


print('Helpers ready.')



# ── Deep-flatten nested dicts/lists in API records ────────────────────────
import ast as _ast

def _deep_flatten(record, prefix=''):
    out = {}
    for k, v in record.items():
        key = f'{prefix}.{k}' if prefix else k
        if isinstance(v, dict):
            out.update(_deep_flatten(v, key))
        elif isinstance(v, list):
            out[key] = json.dumps(v, ensure_ascii=False) if v else None
        elif isinstance(v, str):
            s = v.strip()
            if s.startswith('{') or s.startswith('['):
                try:
                    parsed = json.loads(s)
                except Exception:
                    try: parsed = _ast.literal_eval(s)
                    except Exception: parsed = v
                if isinstance(parsed, dict):
                    out.update(_deep_flatten(parsed, key))
                elif isinstance(parsed, list):
                    out[key] = json.dumps(parsed, ensure_ascii=False) if parsed else None
                else:
                    out[key] = parsed
            else:
                out[key] = v
        else:
            out[key] = v
    return out

def flatten_records(records):
    # Deep-flatten every record; lists become JSON strings
    return [_deep_flatten(r) for r in records]



## 4 · Run all search terms across all categories

Transient server errors (HTTP 500) are retried automatically (fresh session + backoff). A term is checkpointed **only if every category succeeded**; still-failing terms are retried in extra passes at the end and, if they still fail, reported and left out of the checkpoint. Re-running this cell resumes and retries just those.

In [ ]:
# ── Batch scrape all terms (resumable, checkpointed) ─────────────────────
# Writes one JSONL line per completed term to CHECKPOINT_PATH. Re-running this
# cell skips terms already in the checkpoint. Ctrl+C is caught so you keep
# everything scraped so far.
#
# To start fresh: delete the checkpoint file, or change CHECKPOINT_PATH.
import os

CHECKPOINT_PATH = 'anvisa_checkpoint.jsonl'

# Load already-completed terms from previous runs ───────────────────────────
completed_records = []
done_terms = set()
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            done_terms.add(obj['term'])
            completed_records.extend(obj['records'])
    print(f'Resuming: {len(done_terms)} terms already done '
          f'({len(completed_records)} records) loaded from {CHECKPOINT_PATH}')

remaining = [t for t in SEARCH_TERMS if t not in done_terms]
print(f'To do: {len(remaining)} / {len(SEARCH_TERMS)} terms\n')

all_records = list(completed_records)
t_batch_start = time.time()
interrupted = False
failed_terms = []   # terms with >=1 failed category after in-category retries


def _handle_term(search_term, idx, n_total, ckpt, pass_label='', show_eta=False):
    """Scrape one term; checkpoint on full success, else leave it for retry.
    Returns True iff every category succeeded."""
    print(f'{pass_label}[{idx}/{n_total}] Searching: {search_term!r}')
    t_term = time.time()
    term_flat, term_ok, term_total, failed_cats = scrape_term(search_term)

    if term_ok:
        all_records.extend(term_flat)
        # Checkpoint this term (atomic per line, JSONL) -- only when complete
        ckpt.write(json.dumps({'term': search_term, 'records': term_flat},
                              ensure_ascii=False) + '\n')
        ckpt.flush()
        outcome = f'{term_total} records'
    else:
        outcome = (f'{term_total} records but FAILED categories {failed_cats} '
                   f'-- NOT saved, will retry')

    tail = ''
    if show_eta:
        elapsed = time.time() - t_batch_start
        eta = (elapsed / idx) * (n_total - idx)
        tail = f', ETA: {eta/60:.1f} min'
    print(f'  -> {outcome} in {time.time()-t_term:.1f}s '
          f'(total saved: {len(all_records)}{tail})')
    return term_ok


try:
    with open(CHECKPOINT_PATH, 'a', encoding='utf-8') as ckpt:
        # ── Main pass ────────────────────────────────────────────────────
        for idx, search_term in enumerate(remaining, 1):
            if not _handle_term(search_term, idx, len(remaining), ckpt, show_eta=True):
                failed_terms.append(search_term)

        # ── Extra passes over terms that still failed ────────────────────
        for pass_no in range(1, RETRY_PASSES + 1):
            if not failed_terms:
                break
            print(f'\n=== Retry pass {pass_no}/{RETRY_PASSES}: '
                  f'{len(failed_terms)} term(s) to retry (pausing '
                  f'{RETRY_PASS_PAUSE}s first) ===')
            time.sleep(RETRY_PASS_PAUSE)
            retry_now, failed_terms = failed_terms, []
            for idx, search_term in enumerate(retry_now, 1):
                if not _handle_term(search_term, idx, len(retry_now), ckpt,
                                    pass_label=f'[retry {pass_no}] '):
                    failed_terms.append(search_term)
except KeyboardInterrupt:
    interrupted = True
    print(f'\n!! Interrupted. {len(all_records)} records kept; checkpoint '
          f'saved to {CHECKPOINT_PATH}. Re-run this cell to resume.')

status = 'Stopped early' if interrupted else 'Batch done'
print(f'\n=== {status}: {len(all_records)} total records in '
      f'{time.time()-t_batch_start:.1f}s ===')
if failed_terms:
    print(f'\n!! {len(failed_terms)} term(s) still failing after {RETRY_PASSES} '
          f'retry pass(es); NOT written to the checkpoint:')
    for _t in failed_terms:
        print(f'     - {_t!r}')
    print('   Re-run this cell later to try them again.')

# Split into per-category DataFrames ────────────────────────────────────────
df_all = pd.DataFrame(all_records)

def _slice(cat):
    if df_all.empty or '_category' not in df_all.columns:
        return pd.DataFrame()
    return df_all[df_all['_category'] == cat].reset_index(drop=True)

df_alimentos       = _slice('alimentos')
df_cosmeticos_reg  = _slice('cosmeticos_registrados')
df_cosmeticos_reg2 = _slice('cosmeticos_regularizados')
df_medicamentos    = _slice('medicamentos')
df_saneantes       = _slice('saneantes')
df_produtos_saude  = _slice('produtos_saude')

for label, df in [
    ('Alimentos',                df_alimentos),
    ('Cosméticos Registrados',   df_cosmeticos_reg),
    ('Cosméticos Regularizados', df_cosmeticos_reg2),
    ('Medicamentos',             df_medicamentos),
    ('Saneantes',                df_saneantes),
    ('Produtos para Saúde',      df_produtos_saude),
]:
    print(f'{label}: {len(df)} rows, {len(df.columns)} columns')
    if not df.empty:
        display(df.head(3))


## 5 · Post-scraping filters

Applied to `df_all` **before** export:
1. **Word-boundary filter** — drops rows where `_search_term` is embedded inside another word in the product name (e.g. searching `alho` would match `trabALHO` without this).
2. **Skip-terms filter** — drops all rows whose `_search_term` is in `SKIP_TERMS` (e.g. `penicilina`).


In [ ]:
import re as _re

# ── 1. Terms to exclude entirely ─────────────────────────────────────────
# Add terms here to remove all their rows from the final output.
SKIP_TERMS = [
     'colonia',
     'penicilina',
     'trabalho',
     'balm',
     'colônia',
     'balsamo',
     'lemon',
     'cotton',
     'lima',
     'salsa',
     'maravilha',
     'chica',
     'leite vegetal',
     'roma',
     'cheiro verde',
     'lapa',
     'maconha',
     'golfe',
     'novalgina',
     'terramicina',
     'espiga de',
     'loro',
     'angola',
     'burrito',
     'cera cozida',
     'colonia',
     'cpim limao',
     'flor seca',
     'ma is',
     'mmarronnier dinde',
     'pronto alivio'
]
_skip_lower = {t.strip().lower() for t in SKIP_TERMS}

# ── 2. Word-boundary filter ───────────────────────────────────────────────
# Columns to check per category (first column that matches wins).
_NAME_COLS = {
    'alimentos':                ['produto.descricao'],
    'cosmeticos_registrados':   ['nomeProduto'],
    'cosmeticos_regularizados': ['nomeProduto'],
    'medicamentos':             ['produto.nome', 'produto.principioAtivo'],
    'saneantes':                ['nomeProduto'],
    'produtos_saude':           ['produto'],
}

# Unicode-aware: treats accented letters as word characters
_LETTERS = r'a-zA-ZÀ-ɏ'

def _standalone(text, term):
    if not isinstance(text, str) or not term:
        return False
    pat = r'(?<![' + _LETTERS + r'])' + _re.escape(term) + r'(?![' + _LETTERS + r'])'
    return bool(_re.search(pat, text, _re.IGNORECASE))


def _row_passes(row):
    term = str(row.get('_search_term', '')).strip().lower()
    # Skip-terms filter
    if term in _skip_lower:
        return False
    # Word-boundary filter
    cat = row.get('_category', '')
    cols = _NAME_COLS.get(cat, [])
    if not cols:
        return True
    return any(_standalone(str(row.get(c, '')).lower(), term) for c in cols)


before = len(df_all)
df_all = df_all[df_all.apply(_row_passes, axis=1)].reset_index(drop=True)
after  = len(df_all)
print(f'Rows before filters: {before}')
print(f'Rows after  filters: {after}  (removed {before - after})')
if _skip_lower:
    print(f'Skip terms active:   {sorted(_skip_lower)}')

# Rebuild per-category DataFrames from the filtered df_all
df_alimentos      = df_all[df_all['_category'] == 'alimentos'].reset_index(drop=True)
df_cosmeticos_reg = df_all[df_all['_category'] == 'cosmeticos_registrados'].reset_index(drop=True)
df_cosmeticos_reg2= df_all[df_all['_category'] == 'cosmeticos_regularizados'].reset_index(drop=True)
df_medicamentos   = df_all[df_all['_category'] == 'medicamentos'].reset_index(drop=True)
df_saneantes      = df_all[df_all['_category'] == 'saneantes'].reset_index(drop=True)
df_produtos_saude = df_all[df_all['_category'] == 'produtos_saude'].reset_index(drop=True)

for label, df in [
    ('Alimentos',                df_alimentos),
    ('Cosméticos Registrados',   df_cosmeticos_reg),
    ('Cosméticos Regularizados', df_cosmeticos_reg2),
    ('Medicamentos',             df_medicamentos),
    ('Saneantes',                df_saneantes),
    ('Produtos para Saúde',      df_produtos_saude),
]:
    print(f'  {label}: {len(df)} rows')


## 6 · Summary & Excel export


In [ ]:
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

def clean_illegal(df):
    df = df.map(lambda v: ILLEGAL_CHARACTERS_RE.sub('', v) if isinstance(v, str) else v)
    df.columns = [ILLEGAL_CHARACTERS_RE.sub('', str(c)) for c in df.columns]
    return df

out = 'anvisa_plantas_all.xlsx'
with pd.ExcelWriter(out, engine='openpyxl') as w:
    for cat, df in dfs.items():
        if not df.empty:
            df = df.dropna(axis=1, how='all')
            mask = df.apply(lambda c: c.dropna().astype(str).str.strip().isin(['', 'nan', 'None']).all() if c.notna().any() else True)
            df = df.loc[:, ~mask]
            df = clean_illegal(df)
            df.to_excel(w, sheet_name=cat[:31], index=False)

print(f'Saved -> {out}')

In [ ]:
import re as _re2

# ── Excel hard limits ─────────────────────────────────────────────────────
# openpyxl REJECTS these control chars outright (tab/newline/CR are fine).
# ANVISA free-text fields (company names, descriptions) sometimes carry them,
# and a single one anywhere aborts the whole export.
_ILLEGAL_XLSX_RE = _re2.compile(r'[\000-\010\013\014\016-\037]')
EXCEL_MAX_CELL = 32767        # chars per cell
EXCEL_MAX_ROWS = 1_048_575    # data rows per sheet (1,048,576 incl. header)

ALSO_SAVE_CSV = True   # lossless companion copy (Excel truncates huge cells)

_stats = {'ctrl': 0, 'trunc': 0}

def _clean_cell(v):
    """Make one value safe for Excel; count what had to change."""
    if not isinstance(v, str):
        return v
    v2, n = _ILLEGAL_XLSX_RE.subn('', v)
    if n:
        _stats['ctrl'] += n
    if len(v2) > EXCEL_MAX_CELL:
        _stats['trunc'] += 1
        v2 = v2[:EXCEL_MAX_CELL - 20] + '…[TRUNCATED]'
    return v2

def _sanitize(df):
    """Strip illegal chars / over-long cells from every text column."""
    df = df.copy()
    for col in df.columns:
        # pandas 3.x uses 'str' dtype for text; pandas 2.x uses 'object'
        if df[col].dtype == object or pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].map(_clean_cell)
    return df

def _drop_empty_cols(df):
    """Original behaviour: drop columns that are entirely empty."""
    df = df.dropna(axis=1, how='all')
    if df.empty or df.shape[1] == 0:
        return df
    mask = df.apply(lambda c: c.dropna().astype(str).str.strip()
                    .isin(['', 'nan', 'None']).all() if c.notna().any() else True)
    return df.loc[:, ~mask]


dfs = {
    'alimentos':                df_alimentos,
    'cosmeticos_registrados':   df_cosmeticos_reg,
    'cosmeticos_regularizados': df_cosmeticos_reg2,
    'medicamentos':             df_medicamentos,
    'saneantes':                df_saneantes,
    'produtos_saude':           df_produtos_saude,
}

print(f'Total search terms: {len(SEARCH_TERMS)}')
print('-' * 40)
total = 0
for cat, df in dfs.items():
    print(f'  {cat:30s}: {len(df):5d} records')
    total += len(df)
print('-' * 40)
print(f'  {"TOTAL":30s}: {total:5d} records')

out = 'anvisa_plantas_all.xlsx'
written = []
with pd.ExcelWriter(out, engine='openpyxl') as w:
    for cat, df in dfs.items():
        if df.empty:
            continue
        df = _drop_empty_cols(df)
        if df.shape[1] == 0:
            continue
        df = _sanitize(df)
        # Split across sheets if beyond Excel's row ceiling
        n_chunks = (len(df) - 1) // EXCEL_MAX_ROWS + 1
        for i in range(n_chunks):
            chunk = df.iloc[i*EXCEL_MAX_ROWS:(i+1)*EXCEL_MAX_ROWS]
            name = cat[:31] if n_chunks == 1 else f'{cat[:27]}_{i+1}'
            chunk.to_excel(w, sheet_name=name, index=False)
            written.append((name, len(chunk)))
    if not written:
        # openpyxl refuses to save a workbook with zero sheets
        pd.DataFrame({'info': ['No records to export']}).to_excel(
            w, sheet_name='empty', index=False)
        written.append(('empty', 0))

print(f'\nSaved -> {out}')
for name, n in written:
    print(f'    sheet {name:30s} {n:5d} rows')
if _stats['ctrl'] or _stats['trunc']:
    print('\nSanitisation applied so Excel would accept the data:')
    if _stats['ctrl']:
        print(f'  - removed {_stats["ctrl"]} illegal control character(s) from text fields')
    if _stats['trunc']:
        print(f'  - truncated {_stats["trunc"]} cell(s) over {EXCEL_MAX_CELL:,} chars '
              f'(marked "…[TRUNCATED]")')

# Lossless companion — no cell-size or row limits, keeps the full JSON blobs
if ALSO_SAVE_CSV:
    csv_out = 'anvisa_plantas_all.csv'
    df_all.to_csv(csv_out, index=False, encoding='utf-8-sig')
    print(f'\nLossless copy -> {csv_out}  ({len(df_all)} rows, nothing truncated)')


In [ ]:
# ── 7 · Exportar XLSX com UM TERMO DE BUSCA POR LINHA ─────────────────────
# Gera uma planilha-resumo em que cada linha é um termo de busca, com a
# contagem de registros por categoria. Rode DEPOIS do df_all estar montado
# (seção 4 ou seção 7) e, se quiser os números já limpos, depois da seção 5.
import re as _re3
import pandas as pd

OUT_TERMOS = 'anvisa_por_termo.xlsx'

INCLUIR_TERMOS_SEM_RESULTADO = True   # mantém os termos que não acharam nada (total 0)
INCLUIR_NOMES_PRODUTOS       = True   # coluna extra com os nomes dos produtos achados
MAX_NOMES_POR_CELULA         = 60     # corta a lista de nomes para não estourar a célula

CATEGORIAS = ['alimentos', 'cosmeticos_registrados', 'cosmeticos_regularizados',
              'medicamentos', 'saneantes', 'produtos_saude']

# Coluna que carrega o nome do produto em cada categoria (a 1ª que existir vence)
_PROD_COLS = {
    'alimentos':                ['produto.descricao'],
    'cosmeticos_registrados':   ['nomeProduto'],
    'cosmeticos_regularizados': ['nomeProduto'],
    'medicamentos':             ['produto.nome', 'produto.principioAtivo'],
    'saneantes':                ['nomeProduto'],
    'produtos_saude':           ['produto'],
}

_ILEGAL_XLSX = _re3.compile(r'[\000-\010\013\014\016-\037]')
_MAX_CELULA  = 32767


def _norm(s):
    return str(s).strip().lower()


def _limpa(v):
    """Deixa o valor aceitável para o Excel (mesma lógica da célula de export)."""
    if not isinstance(v, str):
        return v
    v = _ILEGAL_XLSX.sub('', v)
    return v[:_MAX_CELULA - 20] + '…[TRUNCADO]' if len(v) > _MAX_CELULA else v


# ── 1. Base: um registro por termo ────────────────────────────────────────
base_terms = [_norm(t) for t in SEARCH_TERMS]
d = df_all.copy()
d['_termo'] = d['_search_term'].map(_norm)

if not INCLUIR_TERMOS_SEM_RESULTADO:
    base_terms = [t for t in base_terms if t in set(d['_termo'])]

resumo = pd.DataFrame({'termo': pd.unique(pd.Series(base_terms + sorted(set(d['_termo']))))})

# ── 2. Nome científico de origem (um termo pode vir de mais de uma espécie) ─
if 'df_terms' in globals():
    _t = df_terms.copy()
    _t['termo'] = _t['search_term'].map(_norm)
    sci = (_t.groupby('termo')['scientific']
             .apply(lambda s: '; '.join(sorted(set(x.strip() for x in s if str(x).strip()))))
             .rename('nome_cientifico'))
    resumo = resumo.merge(sci, on='termo', how='left')
else:
    resumo['nome_cientifico'] = ''

# ── 3. Contagem por categoria + total ─────────────────────────────────────
cont = (d.pivot_table(index='_termo', columns='_category',
                      values='_search_term', aggfunc='size')
          .rename_axis(None, axis=1)
          .rename_axis('termo'))
for cat in CATEGORIAS:                      # garante todas as colunas, mesmo vazias
    if cat not in cont.columns:
        cont[cat] = 0
cont = cont[CATEGORIAS].reset_index()

resumo = resumo.merge(cont, on='termo', how='left')
resumo[CATEGORIAS] = resumo[CATEGORIAS].fillna(0).astype(int)
resumo.insert(2, 'total_registros', resumo[CATEGORIAS].sum(axis=1))
resumo.insert(3, 'categorias_com_resultado', (resumo[CATEGORIAS] > 0).sum(axis=1))

# ── 4. Nomes dos produtos encontrados (opcional) ──────────────────────────
if INCLUIR_NOMES_PRODUTOS:
    def _nome(row):
        for c in _PROD_COLS.get(row.get('_category', ''), []):
            v = row.get(c)
            if isinstance(v, str) and v.strip():
                return v.strip()
        return ''

    nomes_series = d.apply(_nome, axis=1) if len(d) else pd.Series(dtype=str)
    d = d.assign(_nome_produto=nomes_series)

    def _junta(s):
        vistos = sorted({x for x in s if x})
        extra = len(vistos) - MAX_NOMES_POR_CELULA
        txt = ' | '.join(vistos[:MAX_NOMES_POR_CELULA])
        return txt + (f' | …(+{extra})' if extra > 0 else '')

    agreg = (d.groupby('_termo')['_nome_produto']
               .agg([('produtos_unicos', lambda s: len({x for x in s if x})),
                     ('produtos', _junta)])
               .rename_axis('termo').reset_index())
    resumo = resumo.merge(agreg, on='termo', how='left')
    resumo['produtos_unicos'] = resumo['produtos_unicos'].fillna(0).astype(int)
    resumo['produtos'] = resumo['produtos'].fillna('')

# ── 5. Ordena e grava ─────────────────────────────────────────────────────
resumo = (resumo.sort_values(['total_registros', 'termo'], ascending=[False, True])
                .reset_index(drop=True))
resumo = resumo.map(_limpa) if hasattr(resumo, 'map') else resumo.applymap(_limpa)

with pd.ExcelWriter(OUT_TERMOS, engine='openpyxl') as w:
    resumo.to_excel(w, sheet_name='por_termo', index=False)
    ws = w.sheets['por_termo']
    ws.freeze_panes = 'A2'
    larguras = {'termo': 28, 'nome_cientifico': 34, 'total_registros': 16,
                'categorias_com_resultado': 24, 'produtos_unicos': 16, 'produtos': 80}
    for i, col in enumerate(resumo.columns, start=1):
        ws.column_dimensions[ws.cell(row=1, column=i).column_letter].width = \
            larguras.get(col, max(12, min(26, len(col) + 4)))

print(f'Salvo -> {OUT_TERMOS}')
print(f'  {len(resumo)} termos ({(resumo["total_registros"] == 0).sum()} sem nenhum resultado)')
print(f'  {resumo["total_registros"].sum()} registros somados')
display(resumo.head(15))
